# Pytorch Geometric Exercise 1

## Download the data

Download the Cora dataset, the standard benchmark dataset for semi-supervised graph node classification.

In [ ]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='~/A8/PyG Data/Cora', name='Cora')

Get some information about the dataset

In [ ]:
print(
    len(dataset),
    dataset.num_classes,
    dataset.num_features,
)

The dataset contains only 1 single, undirected citation graph

In [ ]:
data = dataset[0]
print(
    data.is_undirected(),
    data.train_mask.sum().item(),
    data.val_mask.sum().item(),
    data.test_mask.sum().item(),
)

The `Data` objects holds a label for each node, and additional node-level attributes: train_mask, val_mask and test_mask, where:

- train_mask denotes against which nodes to train (140 nodes),
- val_mask denotes which nodes to use for validation, e.g., to perform early stopping (500 nodes),
- test_mask denotes against which nodes to test (1000 nodes).

## Create a GCN

See notes for an explanation of the architecture.

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_node_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

## Training the network

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN().to(device)
data = dataset[0].to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f'Accuracy: {acc:.4f}')